# YOLOv11 Bone Fracture Training (Google Colab)
Este notebook unifica el entrenamiento para el dataset original (crudo) y el preprocesado (CLAHE).

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
import yaml
import os

In [ ]:
def unify_humerus_classes(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    if 'humerus' in data['names']:
        humerus_idx = data['names'].index('humerus')
        data['names'].remove('humerus')
        
        with open(yaml_path, 'w') as f:
            yaml.dump(data, f, sort_keys=False)
        
        base_dir = os.path.dirname(yaml_path)
        for split in ['train', 'valid', 'test']:
            label_dir = os.path.join(base_dir, split, 'labels')
            if not os.path.exists(label_dir): continue
            
            for file in os.listdir(label_dir):
                filepath = os.path.join(label_dir, file)
                with open(filepath, 'r') as f:
                    lines = f.readlines()
                
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    cls_id = int(parts[0])
                    if cls_id == humerus_idx:
                        cls_id = data['names'].index('humerus fracture')
                    elif cls_id > humerus_idx:
                        cls_id -= 1
                    parts[0] = str(cls_id)
                    new_lines.append(' '.join(parts) + '\n')
                
                with open(filepath, 'w') as f:
                    f.writelines(new_lines)
        print("Clases unificadas y remapeadas correctamente.")
    else:
        print("La clase 'humerus' ya fue eliminada o no existe.")

In [ ]:
# Cambiar el dataset_path según cuál quieres entrenar:
# dataset_path = "/content/bone-fracture-detection-daoon-1"
dataset_path = "/content/bone-fracture-detection-daoon-1-preprocessed"

yaml_path = os.path.join(dataset_path, "data.yaml")
unify_humerus_classes(yaml_path)

In [ ]:
# Inicializamos y entrenamos YOLO11m
model = YOLO("yolo11m.pt")

results = model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    name="exp_yolo11_colab",
    optimizer="auto",
    patience=30,
    seed=42,
    deterministic=True,
    workers=8, # Óptimo para Colab 8 vCPUs
    device=0
)